# Baseline Models for Ticket Type Classification

This notebook implements baseline machine learning models for the Ticket Type Classification task.

The objective is to automatically classify customer support tickets into predefined categories using the combined ticket subject and description text.

The following baseline models will be evaluated:

1. Logistic Regression
2. Multinomial Naive Bayes

TF-IDF features generated during the text preprocessing stage will be used as input features.

Evaluation metrics:

- Accuracy
- F1-Score (Macro)
- ROC-AUC (One-vs-Rest)

The results obtained in this notebook will serve as a benchmark for comparison with advanced machine learning, deep learning, and transformer-based models.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

In [2]:
# Load datasets
# Features
X_train = pd.read_parquet(
    '../data/processed/ticket_type/X_train.parquet'
)

X_val = pd.read_parquet(
    '../data/processed/ticket_type/X_val.parquet'
)

X_test = pd.read_parquet(
    '../data/processed/ticket_type/X_test.parquet'
)

# Targets
y_train = pd.read_parquet(
    '../data/processed/ticket_type/y_train.parquet'
).squeeze()

y_val = pd.read_parquet(
    '../data/processed/ticket_type/y_val.parquet'
).squeeze()

y_test = pd.read_parquet(
    '../data/processed/ticket_type/y_test.parquet'
).squeeze()

In [5]:
# combine text columns
X_train['combined_text'] = (
    X_train['processed_ticket_subject'] +
    ' ' +
    X_train['processed_description']
)

X_val['combined_text'] = (
    X_val['processed_ticket_subject'] +
    ' ' +
    X_val['processed_description']
)

X_test['combined_text'] = (
    X_test['processed_ticket_subject'] +
    ' ' +
    X_test['processed_description']
)

In [6]:
# verify shapes
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (5928, 3)
X_val: (1270, 3)
X_test: (1271, 3)

y_train: (5928,)
y_val: (1270,)
y_test: (1271,)


In [7]:
# Verify Class Distribution
print(y_train.value_counts())
print("Number of classes:", y_train.nunique())

Ticket Type
Refund request          1226
Technical issue         1223
Cancellation request    1186
Product inquiry         1149
Billing inquiry         1144
Name: count, dtype: int64
Number of classes: 5


In [8]:
# Load Saved TF-IDF Vectorizer
tfidf = joblib.load(
    '../models/tfidf_vectorizer.pkl'
)

print(tfidf)

TfidfVectorizer(max_features=5000, ngram_range=(1, 2))


In [9]:
# Transform Text into TF-IDF Features
X_train_tfidf = tfidf.transform(
    X_train['combined_text']
)

X_val_tfidf = tfidf.transform(
    X_val['combined_text']
)

X_test_tfidf = tfidf.transform(
    X_test['combined_text']
)

# Verify Shapes
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)

Train: (5928, 5000)
Validation: (1270, 5000)
Test: (1271, 5000)


In [10]:
print(X_train_tfidf.shape)
print(y_train.shape)

(5928, 5000)
(5928,)


# Baseline Models

## 1- Logistic Regression Baseline

Logistic Regression is a widely used linear classification algorithm and serves as a strong baseline for text classification tasks. The model is trained using TF-IDF features extracted from the combined ticket subject and ticket description text.

The objective is to establish a benchmark performance for Ticket Type Classification before evaluating more advanced machine learning, deep learning, and transformer-based models.

Model Evaluation Metrics:

- Accuracy
- F1-Score (Macro)
- ROC-AUC (One-vs-Rest)

The trained model will first be evaluated on the validation set and later on the test set for final performance assessment.

In [11]:
# Load and fit model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(
    X_train_tfidf,
    y_train
)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [26]:
# Predictiion
y_val_pred_lr = lr_model.predict(
    X_val_tfidf
)

y_val_pred_prob_lr = lr_model.predict_proba(
    X_val_tfidf
)

In [27]:
# Validation Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

lr_accuracy = accuracy_score(
    y_val,
    y_val_pred_lr
)

lr_f1 = f1_score(
    y_val,
    y_val_pred_lr,
    average='macro'
)

lr_roc_auc = roc_auc_score(
    y_val,
    y_val_pred_prob_lr,
    multi_class='ovr',
    average='macro'
)

print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1-Macro: {lr_f1:.4f}")
print(f"ROC-AUC Score: {lr_roc_auc}")

Accuracy: 0.2087
F1-Macro: 0.2078
ROC-AUC Score: 0.5048549714953604


In [14]:
# Classification report
print(
    classification_report(
        y_val,
        y_val_pred_lr
    )
)

                      precision    recall  f1-score   support

     Billing inquiry       0.21      0.19      0.20       245
Cancellation request       0.21      0.21      0.21       254
     Product inquiry       0.19      0.17      0.18       246
      Refund request       0.20      0.22      0.21       263
     Technical issue       0.24      0.24      0.24       262

            accuracy                           0.21      1270
           macro avg       0.21      0.21      0.21      1270
        weighted avg       0.21      0.21      0.21      1270



In [15]:
print(X_train['combined_text'].head())

0    battery life issue please assist 5 382013 1383...
1    peripheral compatibility issue please assist s...
2    data loss issue please assist hear device supp...
3    network problem noticed software bug app causi...
4    data loss issue please assist var productid do...
Name: combined_text, dtype: object


In [16]:
print(len(tfidf.vocabulary_))

5000


In [17]:
X_train_tfidf.nnz

210965

In [18]:
print(y_train.head(10))

0         Technical issue
1    Cancellation request
2          Refund request
3         Product inquiry
4         Technical issue
5         Billing inquiry
6         Technical issue
7         Billing inquiry
8         Technical issue
9    Cancellation request
Name: Ticket Type, dtype: object


In [19]:
print(
    pd.concat(
        [
            X_train[['processed_ticket_subject']],
            y_train
        ],
        axis=1
    ).head(20)
)

    processed_ticket_subject           Ticket Type
0               battery life       Technical issue
1   peripheral compatibility  Cancellation request
2                  data loss        Refund request
3            network problem       Product inquiry
4                  data loss       Technical issue
5             account access       Billing inquiry
6            network problem       Technical issue
7     product recommendation       Billing inquiry
8             hardware issue       Technical issue
9               software bug  Cancellation request
10    product recommendation       Product inquiry
11     product compatibility       Billing inquiry
12      cancellation request  Cancellation request
13              battery life       Product inquiry
14              software bug       Billing inquiry
15  peripheral compatibility        Refund request
16      installation support  Cancellation request
17  peripheral compatibility       Billing inquiry
18           network problem   

In [20]:
pd.crosstab(
    X_train['processed_ticket_subject'],
    y_train
).head(20)

Ticket Type,Billing inquiry,Cancellation request,Product inquiry,Refund request,Technical issue
processed_ticket_subject,,,,,
account access,73,66,76,78,66
battery life,74,83,72,83,80
cancellation request,51,72,62,66,66
data loss,62,77,68,73,76
delivery problem,84,82,80,79,88
display issue,60,70,55,73,72
hardware issue,71,80,70,94,59
installation support,76,68,62,79,78
network problem,68,67,75,72,90


In [21]:
pd.crosstab(
    X_train['processed_ticket_subject'],
    y_train
).sum(axis=1).sort_values(ascending=False).head(10)

processed_ticket_subject
refund request           418
delivery problem         413
product compatibility    395
software bug             393
battery life             392
product setup            376
payment issue            375
hardware issue           374
network problem          372
installation support     363
dtype: int64

In [22]:
print(y_train.value_counts(normalize=True))

Ticket Type
Refund request          0.206815
Technical issue         0.206309
Cancellation request    0.200067
Product inquiry         0.193826
Billing inquiry         0.192982
Name: proportion, dtype: float64


In [30]:
results = pd.DataFrame({
    'Model': ['Logistic Regression'],
    'Accuracy': [lr_accuracy],
    'F1_Macro': [lr_f1],
    'ROC-AUC' : [lr_roc_auc]
})

results.to_csv('../model_results/baseline_logistic_regression_results.csv', index=False)

In [29]:
results

,Model,Accuracy,F1_Macro,ROC-AUC
0,Logistic Regression,0.208661,0.207785,0.504855


## Baseline Model Analysis (Logistic Regression)

The Logistic Regression baseline achieved relatively low performance:

- Accuracy: 20.87%
- F1-Macro: 20.78%
- ROC-AUC: 50.49%

Further investigation was conducted to verify the preprocessing pipeline, TF-IDF transformation, and train-test split.

A cross-tabulation of ticket subjects against ticket types revealed that the same ticket subjects appeared across multiple target classes with nearly uniform frequency. This indicates weak association between the textual features and the target labels.

As a result, the model performance is close to random guessing (approximately 20% for a 5-class classification problem). This suggests that the dataset provides limited predictive signal for Ticket Type classification using the available text features.

The baseline results will nevertheless serve as a reference point for comparison with more advanced machine learning and deep learning models.

## 2- Multinomial Naive Bayes Baseline

Multinomial Naive Bayes is a probabilistic classification algorithm commonly used for text classification tasks. The model assumes conditional independence among features and is particularly effective when working with TF-IDF representations.

This model serves as a lightweight baseline and provides a comparison against Logistic Regression for Ticket Type Classification.

Evaluation Metrics:

- Accuracy
- F1-Score (Macro)
- ROC-AUC (One-vs-Rest)

The results will be compared with Logistic Regression to identify the stronger baseline model.

In [31]:
# Naive Bayes (Multinomial)
nb_model = MultinomialNB()

nb_model.fit(
    X_train_tfidf,
    y_train
)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [32]:
# Prediction
y_val_pred_nb = nb_model.predict(
    X_val_tfidf
)

y_val_pred_proba_nb = nb_model.predict_proba(
    X_val_tfidf
)

In [34]:
# Validation Metrics
nb_accuracy = accuracy_score(
    y_val,
    y_val_pred_nb
)

nb_f1 = f1_score(
    y_val,
    y_val_pred_nb,
    average='macro'
)

nb_roc_auc = roc_auc_score(
    y_val,
    y_val_pred_proba_nb,
    multi_class='ovr',
    average='macro'
)

print(f"Accuracy: {nb_accuracy:.4f}")
print(f"F1-Macro: {nb_f1:.4f}")
print(f"ROC-AUC: {nb_roc_auc:.4f}")

Accuracy: 0.2071
F1-Macro: 0.2045
ROC-AUC: 0.5099


In [35]:
print(
    classification_report(
        y_val,
        y_val_pred_nb
    )
)

                      precision    recall  f1-score   support

     Billing inquiry       0.19      0.17      0.18       245
Cancellation request       0.20      0.19      0.19       254
     Product inquiry       0.22      0.16      0.19       246
      Refund request       0.19      0.22      0.21       263
     Technical issue       0.23      0.29      0.26       262

            accuracy                           0.21      1270
           macro avg       0.21      0.21      0.20      1270
        weighted avg       0.21      0.21      0.21      1270



In [36]:
from sklearn.metrics import confusion_matrix

cm_nb = confusion_matrix(
    y_val,
    y_val_pred_nb
)

print(cm_nb)

[[42 39 40 64 60]
 [48 48 37 56 65]
 [47 40 40 62 57]
 [42 57 42 58 64]
 [40 62 25 60 75]]


In [39]:
# Model Comparision

results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Multinomial Naive Bayes'
    ],
    'Accuracy': [
        lr_accuracy,
        nb_accuracy
    ],
    'F1_Macro': [
        lr_f1,
        nb_f1
    ],
    'ROC-AUC' : [
        lr_roc_auc,
        nb_roc_auc
    ]
})

results.to_csv('../model_results/baseline_naive_bayes_results.csv', index=False)

In [38]:
results.sort_values(
    by='F1_Macro',
    ascending=False
)

,Model,Accuracy,F1_Macro,ROC-AUC
0,Logistic Regression,0.208661,0.207785,0.504855
1,Multinomial Naive Bayes,0.207087,0.204482,0.509922


## Baseline Model Comparison

Two baseline machine learning models were evaluated using TF-IDF features extracted from the combined ticket subject and ticket description text.
<pre>
| Model | Accuracy | F1-Macro | ROC-AUC |
|--------|----------|----------|----------|
| Logistic Regression | 20.87% | 20.78% | 50.49% |
| Multinomial Naive Bayes | 20.71% | 20.45% | 50.99% |
</pre>
Logistic Regression achieved slightly better performance than Multinomial Naive Bayes. However, both models performed close to the random-guess baseline for a five-class classification problem.

Further exploratory analysis revealed that ticket subjects and descriptions exhibit weak association with the target ticket type labels. Many textual patterns appear across multiple classes with similar frequency, limiting the predictive signal available to the models.

These baseline results will serve as a benchmark for comparison with advanced machine learning, deep learning, and transformer-based approaches in subsequent experiments.

### Test Evaluation on Logistic Regression

In [42]:
# Test predictions
y_test_pred_lr = lr_model.predict(
    X_test_tfidf
)
y_test_pred_prob_lr = lr_model.predict_proba(
    X_test_tfidf
)

# Metrics
lr_test_accuracy = accuracy_score(
    y_test,
    y_test_pred_lr
)

lr_test_f1 = f1_score(
    y_test,
    y_test_pred_lr,
    average='macro'
)

lr_test_roc_auc = roc_auc_score(
    y_test,
    y_test_pred_prob_lr,
    multi_class='ovr',
    average='macro'
)

print(f"Test Accuracy: {lr_test_accuracy:.4f}")
print(f"Test F1-Macro: {lr_test_f1:.4f}")
print(f"ROC-AUC: {lr_test_roc_auc:.4f}")

print(
    classification_report(
        y_test,
        y_test_pred_lr
    )
)

Test Accuracy: 0.2038
Test F1-Macro: 0.2038
ROC-AUC: 0.4924
                      precision    recall  f1-score   support

     Billing inquiry       0.21      0.20      0.20       245
Cancellation request       0.23      0.23      0.23       255
     Product inquiry       0.20      0.19      0.20       246
      Refund request       0.18      0.18      0.18       263
     Technical issue       0.20      0.22      0.21       262

            accuracy                           0.20      1271
           macro avg       0.20      0.20      0.20      1271
        weighted avg       0.20      0.20      0.20      1271



In [ ]:
# save models
from pathlib import Path

Path('../models/baseline').mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    lr_model,
    '../models/baseline/logistic_regression.pkl'
)

joblib.dump(
    nb_model,
    '../models/baseline/multinomial_nb.pkl'
)

print("Models saved successfully.")

['../models/baseline/multinomial_nb.pkl']

# ML Flow logging of baseline models

#### MLflow and DagsHub Configuration

https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.git

In [43]:
import dagshub
import mlflow

dagshub.init(
    repo_owner='armaaz.au.stats',
    repo_name='AI-Powered-Customer-Support-Intelligence-Platform',
    mlflow=True
)

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=5f520cd2-b676-4f5b-918e-a47571b177db&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=0cecf6531a8376b49391cb2045c274663cb78a5995783bb86c0b80729948cccc




Accessing as armaaz.au.stats

Initialized MLflow to track repo "armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform"

Repository armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform initialized!

In [44]:
# Set experiment
mlflow.set_experiment(
    "Ticket_Type_Baseline_Models"
)

2026/06/09 20:07:37 INFO mlflow.tracking.fluent: Experiment with name 'Ticket_Type_Baseline_Models' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/22082c6816f04ecdb54937fc6bc1defa', creation_time=1781015857395, experiment_id='0', last_update_time=1781015857395, lifecycle_stage='active', name='Ticket_Type_Baseline_Models', tags={}, workspace='default'>

In [45]:
# Log Logistic Regression
with mlflow.start_run(run_name="Logistic_Regression"):

    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("max_features", 5000)

    mlflow.log_metric("val_accuracy", lr_accuracy)
    mlflow.log_metric("val_f1_macro", lr_f1)
    mlflow.log_metric("val_roc_auc", lr_roc_auc)

    mlflow.log_metric("test_accuracy", lr_test_accuracy)
    mlflow.log_metric("test_f1_macro", lr_test_f1)
    mlflow.log_metric("test_roc_auc", lr_test_roc_auc)

    mlflow.sklearn.log_model(
        lr_model,
        artifact_path="model"
    )

2026/06/09 20:07:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 20:07:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic_Regression at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/0/runs/f5b3b199fa454015810147e5384a2966
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/0


In [46]:
# Log Naive Bayes
with mlflow.start_run(run_name="Multinomial_Naive_Bayes"):

    mlflow.log_param("model", "Multinomial Naive Bayes")
    mlflow.log_param("max_features", 5000)

    mlflow.log_metric("val_accuracy", nb_accuracy)
    mlflow.log_metric("val_f1_macro", nb_f1)
    mlflow.log_metric("val_roc_auc", nb_roc_auc)

    mlflow.sklearn.log_model(
        nb_model,
        artifact_path="model"
    )

2026/06/09 20:08:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 20:08:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Multinomial_Naive_Bayes at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/0/runs/34f15f13036447a8bcc4a6045844e72a
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/0


## Conclusion

This notebook established baseline performance for the Ticket Type Classification task using traditional machine learning algorithms and TF-IDF text representations.

### Models Evaluated

1. Logistic Regression
2. Multinomial Naive Bayes

### Results Summary
<pre>
| Model | Validation Accuracy | Validation F1-Macro | ValidationROC-AUC |
|---------|---------|---------|---------|
| Logistic Regression | 20.87% | 20.78% | 50.49% |
| Multinomial Naive Bayes | 20.71% | 20.45% | 50.99% |
</pre>
### Key Findings

- Logistic Regression slightly outperformed Multinomial Naive Bayes.
- Both models achieved performance close to the random-guess baseline for a five-class classification problem.
- Investigation of the dataset revealed weak association between textual features and target ticket type labels.
- Similar ticket subjects and descriptions were observed across multiple ticket categories, limiting the predictive signal available to the models.

### Artifacts Generated

- Trained Logistic Regression model
- Trained Multinomial Naive Bayes model
- Baseline performance metrics
- MLflow experiment runs logged to DagsHub

### Next Steps

The next phase of the project will evaluate more advanced machine learning approaches, including:

- Random Forest
- XGBoost
- LightGBM

These models will utilize both text-based and engineered tabular features to determine whether improved predictive performance can be achieved.